# Cross-Lingual Organization Name Normalization - Reference Solution

Given an institution's name in its own language, produce the registry's
canonical English name - for institutions never seen in training.

Three behaviours are mixed into one dataset and have to be separated:
translate, transliterate, and leave alone.

In [ ]:
import collections, os, re
import pandas as pd

PUB = "./dataset/public" if os.path.isdir("./dataset/public") else "./prepared/public"
rd = lambda f: pd.read_csv(f"{PUB}/{f}", keep_default_na=False)
train, test = rd("train.csv"), rd("test.csv")
TOKEN = re.compile(r"\w+", re.UNICODE)
tokens = lambda s: TOKEN.findall(str(s).lower())

print(f"train {len(train):,} pairs   test {len(test):,} pairs")
print(f"languages: {train.lang.nunique()}")
print(f"already English: {(train.source_name == train.target_name).mean():.1%}")

## What the pairs look like

In [ ]:
for _, r in train[train.source_name != train.target_name].head(6).iterrows():
    print(f"{r.lang:>3s}  {r.source_name[:46]:46s} -> {r.target_name[:46]}")

## Scorer

The official metric, so every step below can be measured. A held-out slice of
training is used - and it is held out **by organization**, matching how the
real split was made.

In [ ]:
def token_f1(pred, target):
    p, t = collections.Counter(tokens(pred)), collections.Counter(tokens(target))
    ov = sum((p & t).values())
    if not ov:
        return 0.0
    pr, rc = ov / sum(p.values()), ov / sum(t.values())
    return 2 * pr * rc / (pr + rc)

def score(preds, targets):
    return sum(token_f1(p, t) for p, t in zip(preds, targets)) / len(targets)

ids = sorted(train.pair_id)
cut = set(ids[:int(len(ids) * 0.8)])
fit = train[train.pair_id.isin(cut)]
val = train[~train.pair_id.isin(cut)]
print(f"fit {len(fit):,}  val {len(val):,}")
print("copy source through:", round(score(val.source_name, val.target_name), 4))

## Learn the substitutions

Where source and target have the same token count they can be aligned
positionally, which gives clean evidence. Where they do not, fall back to bag
co-occurrence. Keep tables per language and pooled - French evidence should not
fire on Japanese, but pooled evidence covers the long tail.

In [ ]:
MIN_EVIDENCE = 2      # a substitution needs this many aligned observations
RARE_BELOW = 3        # tokens rarer than this are treated as proper nouns

per_lang = collections.defaultdict(lambda: collections.defaultdict(collections.Counter))
pooled   = collections.defaultdict(collections.Counter)
cooccur  = collections.defaultdict(collections.Counter)
freq     = collections.Counter()

for src, lang, tgt in zip(fit.source_name, fit.lang, fit.target_name):
    s, t = tokens(src), tokens(tgt)
    freq.update(s)
    if len(s) == len(t):                 # positionally alignable
        for a, b in zip(s, t):
            per_lang[lang][a][b] += 1
            pooled[a][b] += 1
    else:                                # fall back to bag co-occurrence
        for a in set(s):
            for b in set(t):
                cooccur[a][b] += 1

def best(counter, floor):
    top = counter.most_common(1)
    return top[0][0] if top and top[0][1] >= floor else None

lang_tbl   = {l: {w: best(c, MIN_EVIDENCE) for w, c in d.items()} for l, d in per_lang.items()}
pooled_tbl = {w: best(c, MIN_EVIDENCE) for w, c in pooled.items()}
cooc_tbl   = {w: best(c, MIN_EVIDENCE + 1) for w, c in cooccur.items()}
print(f"{sum(1 for v in pooled_tbl.values() if v):,} pooled substitutions, {len(lang_tbl)} languages")

## The copy-through rule

A token seen once or twice in 41,000 pairs is almost certainly a place or a
person, not vocabulary. Forcing it through the table corrupts a name that was
already right. Below, `RARE_BELOW` controls that and it is worth several
points.

In [ ]:
def translate(src, lang, rare_below=3):
    out = []
    for w in tokens(src):
        sub = (lang_tbl.get(lang, {}).get(w)
               or pooled_tbl.get(w)
               or (w if freq[w] < rare_below else cooc_tbl.get(w)))
        out.append(sub or w)
    return " ".join(out)

for rb in (0, 3, 10):
    preds = [translate(s, l, rb) for s, l in zip(val.source_name, val.lang)]
    print(f"rare_below={rb:2d}  {score(preds, val.target_name):.4f}")

## Refit on all training pairs and submit

In [ ]:
# refit the tables on every training pair before predicting
for src, lang, tgt in zip(val.source_name, val.lang, val.target_name):
    s, t = tokens(src), tokens(tgt)
    freq.update(s)
    if len(s) == len(t):
        for a, b in zip(s, t):
            per_lang[lang][a][b] += 1
            pooled[a][b] += 1
lang_tbl   = {l: {w: best(c, MIN_EVIDENCE) for w, c in d.items()} for l, d in per_lang.items()}
pooled_tbl = {w: best(c, MIN_EVIDENCE) for w, c in pooled.items()}

outputs = [translate(s, l) for s, l in zip(test.source_name, test.lang)]
os.makedirs("./working", exist_ok=True)
pd.DataFrame({"pair_id": test.pair_id, "output": outputs}).to_csv(
    "./working/submission.csv", index=False)
print(f"wrote {len(outputs):,} rows")
pd.DataFrame({"source": list(test.source_name[:5]), "predicted": outputs[:5]})

## On the real test set

This solver scores **0.494** token F1. Copying the source through unchanged
scores 0.428, so the learned vocabulary is worth about seven points on top of
a baseline that is already high because 38 percent of pairs need no change.

The remaining gap is largely not learnable under the rules. Turning
"Universiteit Antwerpen" into "University of Antwerp" needs to know that
Antwerpen is Antwerp, and no amount of pattern induction over these pairs
supplies that unless the city happens to appear in them.